# ACDC IOI Cross-Model Validation — Stage 1

Edge-level ACDC-style IOI discovery on 4 TransformerLens-loadable model variants:
- GPT-J 6B
- Qwen-2.5-7B-Instruct
- Qwen-2.5-14B-Instruct
- Gemma-2-9B-IT

These are the four variants whose data ship in `data/acdc_ioi_cross_model/`. The two additional registry entries (`llama-3.1-8b-instruct`, `olmo-2-13b-instruct`) are exploratory-only — they are not part of the submitted ACDC evidence (Appendix F.5 / Table F4) and the corresponding 30_patching outputs are not bundled in the supplementary archive.

N_PROMPTS = 50 IOI prompts per model.

Phases per model:
- Phase 10: Collection (IOI prompts, caches, solvability check)
- Phase 20: Scoring (gradient proxy importance + perturbation + cell classification)
- Phase 30: Dose response (9 groups, ratio sweep)
- Phase 35: Recoverability (threshold rules, baselines)
- Phase 36: Discovery summaries

All outputs saved to `/content/drive/MyDrive/WCC/tasks/acdc_ioi_cross_model/`

Uses TransformerLens HookedTransformer for all models.

Edge attribution: AtP*-style gradient proxy (Phase 20).
Edge intervention: source-head hook_z replacement (Phase 30+).
Unit: directed edge between attention heads (src_layer < dst_layer).

Estimated runtime: ~4h on A100.

In [ ]:
# Cell 1: Install (ランタイム再起動！)
!pip install -q transformer-lens
!pip install --force-reinstall numpy pandas typing-extensions scipy
print('★★★ Runtime → Restart runtime → Cell 2から実行 ★★★')

In [ ]:
import os
from google.colab import drive, runtime

# HF_TOKEN — DO NOT REMOVE
os.environ['HF_TOKEN'] = '<YOUR_HF_TOKEN>'
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF_TOKEN set + logged in.')

drive.mount('/content/drive')

# Cell 2: Constants, Config, Model Registry

import gc
import json
import math
import os
import time
import warnings
from collections import defaultdict
from datetime import datetime
from typing import Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr, rankdata

warnings.filterwarnings('ignore', category=FutureWarning)

assert torch.cuda.is_available(), 'GPU runtime required -- select A100'
DEVICE = 'cuda'

# ============================================================
# Project paths
# ============================================================
DRIVE_BASE = '/content/drive/MyDrive/WCC/tasks/acdc_ioi_cross_model'
os.makedirs(DRIVE_BASE, exist_ok=True)

# ============================================================
# Experiment constants
# ============================================================
N_PROMPTS = 50
RATIOS = [0.01, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.30, 0.50]
CACHE_BATCH = 16       # Batch size for hook_z caching
INTERVENTION_BATCH = 8 # Batch size for dose-response / faithfulness
FAITHFULNESS_BUDGETS = [10, 25, 50, 100, 200, 500]

# Threshold rules for Phase 35
THRESHOLD_RULES = {
    'mean_1.5sd': lambda vals: float(vals.mean() + 1.5 * vals.std()),
    'mean_2.0sd': lambda vals: float(vals.mean() + 2.0 * vals.std()),
    'mean_2.5sd': lambda vals: float(vals.mean() + 2.5 * vals.std()),
    'p95': lambda vals: float(np.percentile(vals, 95)),
}

# Baselines for Phase 35/36
BASELINES = [
    ('standard',    'cell_classification.csv'),
    ('celld',       'cell_classification.csv'),
    ('absld',       'cell_classification_abs.csv'),
    ('absld_celld', 'cell_classification_abs.csv'),
]

# ============================================================
# Model registry -- 4 ACDC-shipped variants + 2 exploratory entries
# ============================================================
MODEL_REGISTRY = {
    'gpt-j-6b': {
        'tl_name': 'EleutherAI/gpt-j-6B',
        'hf_name': None,
        'n_layers': 28, 'n_heads': 16,
        'd_model': 4096, 'd_head': 256,
        'dtype': torch.float16,
        'has_gt_heads': False,
        'notes': 'TransformerLens direct load. No Wang et al. GT available.',
    },
    'llama-3.1-8b-instruct': {
        'tl_name': 'meta-llama/Llama-3.1-8B-Instruct',
        'hf_name': 'meta-llama/Llama-3.1-8B-Instruct',
        'n_layers': 32, 'n_heads': 32,
        'd_model': 4096, 'd_head': 128,
        'dtype': torch.float16,
        'has_gt_heads': False,
        'notes': 'May need hf_model wrapping if TL direct fails.',
    },
    'qwen-2.5-7b-instruct': {
        'tl_name': 'Qwen/Qwen2.5-7B-Instruct',
        'hf_name': 'Qwen/Qwen2.5-7B-Instruct',
        'n_layers': 28, 'n_heads': 28,
        'd_model': 3584, 'd_head': 128,
        'dtype': torch.float16,
        'has_gt_heads': False,
        'notes': 'May need hf_model wrapping if TL direct fails.',
    },
    'qwen-2.5-14b-instruct': {
        'tl_name': 'Qwen/Qwen2.5-14B-Instruct',
        'hf_name': 'Qwen/Qwen2.5-14B-Instruct',
        'n_layers': 48, 'n_heads': 40,
        'd_model': 5120, 'd_head': 128,
        'dtype': torch.float16,
        'has_gt_heads': False,
        'notes': 'May need hf_model wrapping if TL direct fails.',
    },
    'gemma-2-9b-it': {
        'tl_name': 'google/gemma-2-9b-it',
        'hf_name': 'google/gemma-2-9b-it',
        'n_layers': 42, 'n_heads': 16,
        'd_model': 3584, 'd_head': 256,
        'dtype': torch.float16,
        'has_gt_heads': False,
        'notes': 'May need hf_model wrapping if TL direct fails.',
    },
    'olmo-2-13b-instruct': {
        'tl_name': 'allenai/OLMo-2-1124-13B-Instruct',
        'hf_name': 'allenai/OLMo-2-1124-13B-Instruct',
        'n_layers': 40, 'n_heads': 40,
        'd_model': 5120, 'd_head': 128,
        'dtype': torch.float16,
        'has_gt_heads': False,
        'notes': 'May need hf_model wrapping if TL direct fails.',
    },
}

# The four model variants whose 30_patching outputs ship in
# data/acdc_ioi_cross_model/. llama-3.1-8b-instruct and olmo-2-13b-instruct
# remain in MODEL_REGISTRY for completeness (exploratory loads only) and
# are NOT part of the submitted ACDC evidence (Appendix F.5 / Table F4).
MODEL_ORDER = [
    'gpt-j-6b',
    'gemma-2-9b-it',
    'qwen-2.5-7b-instruct',
    'qwen-2.5-14b-instruct',
]

print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'Drive base: {DRIVE_BASE}')
print(f'Models ({len(MODEL_ORDER)}): {MODEL_ORDER}')
print(f'N_PROMPTS: {N_PROMPTS}')

In [ ]:
# Cell 3: IOI Prompt Generation + Utility Functions

# ============================================================
# Logging
# ============================================================
def log(msg):
    print(f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] {msg}', flush=True)

def print_vram(prefix=''):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        peak = torch.cuda.max_memory_allocated() / 1024**3
        log(f'{prefix} VRAM allocated={alloc:.2f}GB peak={peak:.2f}GB')

# ============================================================
# Progress tracking
# ============================================================
def load_progress():
    path = os.path.join(DRIVE_BASE, 'progress.json')
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {'completed_models': {}, 'current_model': None, 'current_phase': None}

def save_progress(progress):
    progress['timestamp'] = datetime.now().isoformat()
    path = os.path.join(DRIVE_BASE, 'progress.json')
    with open(path, 'w') as f:
        json.dump(progress, f, indent=2)

# ============================================================
# IOI prompt generation
# ============================================================
IOI_TEMPLATES = [
    'Then, {A} and {B} went to the {PLACE}. {C} gave a {OBJECT} to',
    'Then, {A} and {B} had a meeting at the {PLACE}. {C} handed a {OBJECT} to',
    'After that, {A} and {B} visited the {PLACE}. {C} passed the {OBJECT} to',
    'Next, {A} and {B} stopped by the {PLACE}. {C} offered a {OBJECT} to',
    'Then, {A} and {B} arrived at the {PLACE}. {C} sent the {OBJECT} to',
]

IOI_NAMES = [
    'Alice', 'Bob', 'Charlie', 'David', 'Emma', 'Frank', 'Grace',
    'Henry', 'Irene', 'Jack', 'Karen', 'Liam', 'Mia', 'Nathan',
    'Olivia', 'Peter', 'Quinn', 'Rachel', 'Sam', 'Tom', 'Uma',
    'Victor', 'Wendy', 'Xavier', 'Yara', 'Zack', 'Diana', 'Eric',
    'Fiona', 'George', 'Helen', 'Ivan', 'Julia', 'Kevin', 'Lucy',
    'Mark', 'Nina', 'Oscar', 'Paula', 'Rosa', 'Steve', 'Tina',
]

IOI_PLACES = [
    'store', 'park', 'school', 'office', 'library', 'restaurant',
    'hospital', 'airport', 'station', 'market', 'museum', 'hotel',
]

IOI_OBJECTS = [
    'book', 'letter', 'gift', 'message', 'ticket', 'key',
    'drink', 'card', 'note', 'phone', 'bag', 'box',
]


def generate_ioi_prompts(n_prompts, seed=42):
    """Generate N IOI prompts with clean and corrupt versions.

    Clean: '...{S} and {IO} ... {S} gave ... to' -> expects IO
    Corrupt (ABB): '...{IO} and {IO} ... {IO} gave ... to' -> corrupted pattern

    IOI convention: IO = indirect object (correct answer),
    S = subject (repeated name).
    """
    rng = np.random.RandomState(seed)
    prompts = []
    for i in range(n_prompts):
        name_indices = rng.choice(len(IOI_NAMES), size=2, replace=False)
        io_name = IOI_NAMES[name_indices[0]]
        s_name = IOI_NAMES[name_indices[1]]
        template = IOI_TEMPLATES[i % len(IOI_TEMPLATES)]
        place = IOI_PLACES[rng.randint(len(IOI_PLACES))]
        obj = IOI_OBJECTS[rng.randint(len(IOI_OBJECTS))]
        clean_text = template.format(
            A=s_name, B=io_name, C=s_name, PLACE=place, OBJECT=obj)
        corrupt_text = template.format(
            A=io_name, B=io_name, C=io_name, PLACE=place, OBJECT=obj)
        prompts.append({
            'prompt_id': i, 'io_name': io_name, 's_name': s_name,
            'clean_text': clean_text, 'corrupt_text': corrupt_text,
            'template_id': i % len(IOI_TEMPLATES),
            'place': place, 'object': obj,
        })
    return pd.DataFrame(prompts)


# ============================================================
# IOI task score
# ============================================================
def get_ioi_token_ids(model, prompts_df):
    """Get IO and S token IDs for task score computation."""
    io_ids, s_ids = [], []
    for _, row in prompts_df.iterrows():
        io_tok = model.to_tokens(
            f" {row['io_name']}", prepend_bos=False)[0, 0].item()
        s_tok = model.to_tokens(
            f" {row['s_name']}", prepend_bos=False)[0, 0].item()
        io_ids.append(io_tok)
        s_ids.append(s_tok)
    return io_ids, s_ids


def ioi_task_score_batch(logits, io_ids, s_ids):
    """IOI task score = logit[IO] - logit[S] at last position.
    logits: (batch, seq, vocab). Returns list of floats."""
    scores = []
    for b in range(logits.shape[0]):
        scores.append(float(
            logits[b, -1, io_ids[b]] - logits[b, -1, s_ids[b]]))
    return scores


def ioi_task_score(logits, io_id, s_id):
    """Single-prompt IOI task score."""
    return float(logits[0, -1, io_id] - logits[0, -1, s_id])


# ============================================================
# Edge construction
# ============================================================
def build_edges_df(n_layers, n_heads, gt_set=None, gt_sub=None):
    """Build candidate edges DataFrame (src_layer < dst_layer)."""
    if gt_set is None:
        gt_set = set()
    if gt_sub is None:
        gt_sub = {}
    edges = []
    eid = 0
    for sl in range(n_layers):
        for sh in range(n_heads):
            for dl in range(sl + 1, n_layers):
                for dh in range(n_heads):
                    edges.append({
                        'edge_id': eid,
                        'src_layer': sl, 'src_head': sh,
                        'dst_layer': dl, 'dst_head': dh,
                        'src_node_id': f'L{sl}H{sh}',
                        'dst_node_id': f'L{dl}H{dh}',
                        'src_in_gt': int((sl, sh) in gt_set),
                        'dst_in_gt': int((dl, dh) in gt_set),
                        'src_gt_subclass': gt_sub.get((sl, sh), ''),
                        'dst_gt_subclass': gt_sub.get((dl, dh), ''),
                    })
                    eid += 1
    return pd.DataFrame(edges)


# ============================================================
# Model loading via TransformerLens
# ============================================================
def load_tl_model(model_name):
    """Load model via TransformerLens HookedTransformer.
    Tries direct from_pretrained first; falls back to hf_model wrapping."""
    from transformer_lens import HookedTransformer
    from transformers import AutoModelForCausalLM, AutoTokenizer

    cfg = MODEL_REGISTRY[model_name]
    tl_name = cfg['tl_name']
    dtype = cfg['dtype']

    log(f'Loading {model_name} via TransformerLens...')
    print_vram('[before load]')

    # Try direct TransformerLens load
    try:
        model = HookedTransformer.from_pretrained(
            tl_name, device=DEVICE, dtype=dtype)
        model.eval()
        log(f'  Direct TL load succeeded: {tl_name}')
        print_vram('[after load]')
        return model
    except Exception as e:
        log(f'  Direct TL load failed: {e}')

    # Fallback: HF model + tokenizer wrapping
    hf_name = cfg.get('hf_name') or tl_name
    log(f'  Trying hf_model wrapping for {hf_name}...')
    hf_model = AutoModelForCausalLM.from_pretrained(
        hf_name, torch_dtype=dtype, device_map='auto',
        trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(
        hf_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = HookedTransformer.from_pretrained(
        hf_name, hf_model=hf_model, tokenizer=tokenizer,
        device=DEVICE, dtype=dtype)
    model.eval()
    del hf_model
    gc.collect()
    torch.cuda.empty_cache()
    log(f'  HF-wrapped TL load succeeded: {hf_name}')
    print_vram('[after load]')
    return model


def unload_model(model):
    """Free model VRAM completely."""
    print_vram('[before unload]')
    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    log('Model unloaded.')
    print_vram('[after unload]')


# ============================================================
# Cell classification
# ============================================================
def classify_cells(imp_values, pert_values):
    """Median-split cell classification."""
    med_imp = np.median(imp_values)
    med_pert = np.median(pert_values)
    cells = []
    for imp, pert in zip(imp_values, pert_values):
        hi_i = imp >= med_imp
        hi_p = pert >= med_pert
        if hi_i and hi_p: cells.append('A')
        elif hi_i and not hi_p: cells.append('B')
        elif not hi_i and hi_p: cells.append('C')
        else: cells.append('D')
    return cells


# ============================================================
# Dose response orderings
# ============================================================
def build_orderings(cell_df):
    """Build 9 orderings from cell classification."""
    imp = {int(r['edge_id']): abs(r['imp_value'])
           for _, r in cell_df.iterrows()}
    pert = {int(r['edge_id']): r['pert_value']
            for _, r in cell_df.iterrows()}
    c_ids = cell_df[cell_df['cell'] == 'C']['edge_id'].tolist()
    d_ids = cell_df[cell_df['cell'] == 'D']['edge_id'].tolist()
    all_ids = c_ids + d_ids
    orderings = {
        'C_imp_asc': sorted(c_ids, key=lambda e: imp.get(e, 0)),
        'C_pert_asc': sorted(c_ids, key=lambda e: pert.get(e, 0)),
        'C_pert_desc': sorted(
            c_ids, key=lambda e: pert.get(e, 0), reverse=True),
        'D_imp_asc': sorted(d_ids, key=lambda e: imp.get(e, 0)),
        'D_pert_asc': sorted(d_ids, key=lambda e: pert.get(e, 0)),
        'D_pert_desc': sorted(
            d_ids, key=lambda e: pert.get(e, 0), reverse=True),
        'std_imp_asc': sorted(all_ids, key=lambda e: imp.get(e, 0)),
    }
    imp_arr = np.array([imp.get(e, 0) for e in all_ids])
    pert_arr = np.array([pert.get(e, 0) for e in all_ids])
    if (len(all_ids) > 0 and imp_arr.std() > 0 and pert_arr.std() > 0):
        z_imp = (imp_arr - imp_arr.mean()) / (imp_arr.std() + 1e-10)
        z_pert = (pert_arr - pert_arr.mean()) / (pert_arr.std() + 1e-10)
        orderings['diag_zscore_asc'] = [
            all_ids[i] for i in np.argsort(z_imp + z_pert)]
        rank_imp = rankdata(imp_arr)
        rank_pert = rankdata(pert_arr)
        orderings['diag_rank_asc'] = [
            all_ids[i] for i in np.argsort(rank_imp + rank_pert)]
    else:
        orderings['diag_zscore_asc'] = orderings['std_imp_asc'][:]
        orderings['diag_rank_asc'] = orderings['std_imp_asc'][:]
    return orderings


log('Cell 3 complete: utility functions defined.')

In [ ]:
# Cell 4: Phase 10 -- Collection (batched hook_z caching)

def run_phase_10(model_name, model, prompts_df):
    """Phase 10: Collection -- caches, solvability, edges.
    Uses batched hook_z caching for speed."""
    cfg = MODEL_REGISTRY[model_name]
    n_layers = cfg['n_layers']
    n_heads = cfg['n_heads']
    d_model = cfg['d_model']
    d_head = cfg['d_head']

    phase_dir = os.path.join(DRIVE_BASE, '10_collection', model_name)
    os.makedirs(phase_dir, exist_ok=True)
    config_path = os.path.join(phase_dir, 'config.json')

    log(f'=== Phase 10: Collection [{model_name}] ===')
    t0 = time.time()

    # Resume check
    if os.path.exists(config_path):
        with open(config_path) as f:
            existing = json.load(f)
        if existing.get('solvable') is False:
            log(f'  Phase 10 shows {model_name} not solvable. Skipping.')
            return False
        log(f'  Phase 10 already complete for {model_name}, skipping.')
        return True

    prompts_df.to_csv(os.path.join(phase_dir, 'prompts.csv'), index=False)
    io_ids, s_ids = get_ioi_token_ids(model, prompts_df)

    # --- IOI solvability check (batched) ---
    clean_scores = []
    texts = list(prompts_df['clean_text'])
    for bs in range(0, len(texts), CACHE_BATCH):
        batch_texts = texts[bs:bs + CACHE_BATCH]
        batch_io = io_ids[bs:bs + CACHE_BATCH]
        batch_s = s_ids[bs:bs + CACHE_BATCH]
        tokens = model.to_tokens(batch_texts, prepend_bos=True)
        with torch.no_grad():
            logits = model(tokens.to(DEVICE))
        clean_scores.extend(
            ioi_task_score_batch(logits, batch_io, batch_s))

    mean_ld = float(np.mean(clean_scores))
    log(f'  IOI solvability: mean logit_diff = {mean_ld:.4f}')

    if mean_ld <= 0:
        log(f'  FAILED: {model_name} does not solve IOI. SKIPPING.')
        config = {
            'phase': '10_collection', 'model': model_name,
            'solvable': False, 'mean_logit_diff': mean_ld,
            'completed_at': datetime.now().isoformat(),
        }
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)
        return False

    # Architecture
    arch = {
        'model': model_name, 'tl_name': cfg['tl_name'],
        'n_layers': n_layers, 'n_heads': n_heads,
        'd_model': d_model, 'd_head': d_head, 'device': DEVICE,
    }
    with open(os.path.join(phase_dir, 'architecture.json'), 'w') as f:
        json.dump(arch, f, indent=2)

    # Candidate nodes + edges
    nodes = [{'node_id': f'L{l}H{h}', 'layer': l, 'head': h,
              'node_type': 'attn_head'}
             for l in range(n_layers) for h in range(n_heads)]
    pd.DataFrame(nodes).to_csv(
        os.path.join(phase_dir, 'candidate_nodes.csv'), index=False)

    edges_df = build_edges_df(n_layers, n_heads)
    edges_df.to_csv(
        os.path.join(phase_dir, 'candidate_edges.csv'), index=False)
    log(f'  Candidate edges: {len(edges_df)}')

    # --- Batched hook_z caching ---
    for condition, text_col in [
        ('clean', 'clean_text'), ('corrupt', 'corrupt_text')
    ]:
        log(f'  Caching {condition} hook_z (batched, bs={CACHE_BATCH})...')
        cache_dict = {layer: [] for layer in range(n_layers)}
        all_texts = list(prompts_df[text_col])

        for bs in range(0, len(all_texts), CACHE_BATCH):
            batch_texts = all_texts[bs:bs + CACHE_BATCH]
            tokens = model.to_tokens(
                batch_texts, prepend_bos=True).to(DEVICE)
            with torch.no_grad():
                _, cache = model.run_with_cache(tokens)
            for layer in range(n_layers):
                # hook_z: (batch, seq, n_heads, d_head) -> last token
                z = cache[
                    f'blocks.{layer}.attn.hook_z'
                ][:, -1, :, :].cpu()  # (batch, n_heads, d_head)
                cache_dict[layer].append(z)
            del cache
            torch.cuda.empty_cache()

        stacked = {
            layer: torch.cat(cache_dict[layer], dim=0)
            for layer in range(n_layers)
        }  # {layer: (n_prompts, n_heads, d_head)}
        torch.save(stacked,
                   os.path.join(phase_dir, f'{condition}_hook_z.pt'))
        log(f'    Saved {condition}_hook_z.pt '
            f'shape[0]={stacked[0].shape}')
        del stacked, cache_dict
        gc.collect()
        torch.cuda.empty_cache()

    # Save task scores + token IDs
    torch.save(
        {'scores': clean_scores, 'io_ids': io_ids, 's_ids': s_ids},
        os.path.join(phase_dir, 'task_scores.pt'))

    config = {
        'phase': '10_collection', 'model': model_name, 'task': 'ioi',
        'n_prompts': len(prompts_df), 'n_edges': len(edges_df),
        'solvable': True, 'mean_logit_diff': mean_ld,
        'source_head_filtering': False,
        'hook_cached': 'hook_z (per-head, d_head)',
        'cache_batch_size': CACHE_BATCH,
        'prompts_source': 'generated with seed=42',
        'completed_at': datetime.now().isoformat(),
        'elapsed_sec': round(time.time() - t0, 1),
    }
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    log(f'  Phase 10 complete: {time.time() - t0:.1f}s')
    return True


log('Cell 4 defined: Phase 10 (Collection).')

In [ ]:
# Cell 5: Phase 20 -- Scoring (gradient proxy + perturbation)

def run_phase_20(model_name, model):
    """Phase 20: Edge scoring via gradient proxy.

    Signed importance: grad_at_dst_input . (clean_src_output - corrupt_src_output)
    W_O projection: src_delta_dmodel = src_z_delta @ model.W_O[layer, head]
    Perturbation: source-side L2 of projected delta.

    Batches multiple prompts per dst_layer gradient computation.
    """
    cfg = MODEL_REGISTRY[model_name]
    n_layers = cfg['n_layers']
    n_heads = cfg['n_heads']
    d_model = cfg['d_model']

    phase_dir = os.path.join(DRIVE_BASE, '20_scoring', model_name)
    os.makedirs(phase_dir, exist_ok=True)
    coll_dir = os.path.join(DRIVE_BASE, '10_collection', model_name)
    config_path = os.path.join(phase_dir, 'config.json')

    log(f'=== Phase 20: Scoring [{model_name}] ===')
    t0 = time.time()

    if os.path.exists(config_path):
        log(f'  Phase 20 already complete, skipping.')
        return

    prompts_df = pd.read_csv(os.path.join(coll_dir, 'prompts.csv'))
    edges_df = pd.read_csv(os.path.join(coll_dir, 'candidate_edges.csv'))
    clean_z = torch.load(
        os.path.join(coll_dir, 'clean_hook_z.pt'), map_location='cpu')
    corrupt_z = torch.load(
        os.path.join(coll_dir, 'corrupt_hook_z.pt'), map_location='cpu')
    task_data = torch.load(
        os.path.join(coll_dir, 'task_scores.pt'), map_location='cpu')
    io_ids, s_ids = task_data['io_ids'], task_data['s_ids']
    n_prompts = len(prompts_df)
    n_edges = len(edges_df)

    # Pre-compute W_O per (layer, head)
    log('  Pre-computing W_O projections...')
    W_O = {}
    for layer in range(n_layers):
        for head in range(n_heads):
            W_O[(layer, head)] = model.W_O[layer, head].cpu().float()

    # Source deltas in d_model space
    log('  Computing source deltas...')
    src_delta_dmodel = {}
    for layer in range(n_layers):
        delta_z = clean_z[layer] - corrupt_z[layer]
        for head in range(n_heads):
            dz = delta_z[:, head, :].float()
            src_delta_dmodel[(layer, head)] = dz @ W_O[(layer, head)]

    # Gradient at dst_layer resid_pre -- batch prompts per layer
    log('  Computing dst gradients (batched)...')
    dst_grads = {}
    all_texts = list(prompts_df['clean_text'])

    for dst_layer in range(1, n_layers):
        layer_grads = []
        for bs in range(0, n_prompts, CACHE_BATCH):
            batch_texts = all_texts[bs:bs + CACHE_BATCH]
            batch_io = io_ids[bs:bs + CACHE_BATCH]
            batch_s = s_ids[bs:bs + CACHE_BATCH]
            tokens = model.to_tokens(
                batch_texts, prepend_bos=True).to(DEVICE)
            batch_size = tokens.shape[0]

            grad_holder = {}
            def _hook(value, hook, _h=grad_holder):
                value.requires_grad_(True)
                value.retain_grad()
                _h['val'] = value
                return value

            hook_name = f'blocks.{dst_layer}.hook_resid_pre'
            try:
                with torch.enable_grad():
                    logits = model.run_with_hooks(
                        tokens, fwd_hooks=[(hook_name, _hook)])
                    # Sum task scores across batch for single backward
                    score_sum = sum(
                        logits[b, -1, batch_io[b]]
                        - logits[b, -1, batch_s[b]]
                        for b in range(batch_size))
                    score_sum.backward()

                if ('val' in grad_holder
                        and grad_holder['val'].grad is not None):
                    # grad shape: (batch, seq, d_model) -> last token
                    g = grad_holder['val'].grad[
                        :, -1, :].detach().cpu().float()
                    for b in range(batch_size):
                        layer_grads.append(g[b])
                else:
                    for _ in range(batch_size):
                        layer_grads.append(
                            torch.full((d_model,), float('nan')))
            except Exception as e:
                log(f'    WARNING: grad failed dst_layer={dst_layer} '
                    f'batch={bs}: {e}')
                for _ in range(len(batch_texts)):
                    layer_grads.append(
                        torch.full((d_model,), float('nan')))
            finally:
                model.zero_grad()
                torch.cuda.empty_cache()

        dst_grads[dst_layer] = torch.stack(layer_grads)
        if (dst_layer % 5 == 0) or (dst_layer == n_layers - 1):
            log(f'    dst_layer={dst_layer}/{n_layers-1} done')

    # Compute edge importance and perturbation
    log('  Computing edge scores...')
    # --- Vectorized edge scoring ---
    # Step 1: Per source-head (n_heads total, not n_edges)
    src_pert = {}
    src_pv = {}
    for sl in range(cfg['n_layers']):
        for sh in range(cfg['n_heads']):
            d = src_delta_dmodel[(sl, sh)]
            src_pert[(sl, sh)] = float(d.norm(dim=1).mean().detach())
            clean_dm = clean_z[sl][:, sh, :].float() @ W_O[(sl, sh)]
            corrupt_dm = corrupt_z[sl][:, sh, :].float() @ W_O[(sl, sh)]
            mean_dm = clean_dm.mean(dim=0, keepdim=True)
            src_pv[(sl, sh)] = {
                'pert_resample': float((clean_dm - corrupt_dm).norm(dim=1).mean().detach()),
                'pert_mean': float((clean_dm - mean_dm).norm(dim=1).mean().detach()),
                'pert_zero': float(clean_dm.norm(dim=1).mean().detach()),
            }
    log(f'    Source-head quantities: {cfg["n_layers"]*cfg["n_heads"]} heads')

    # Step 2: Importance per (src_head, dst_layer)
    imp_lookup = {}
    for dl in range(1, cfg['n_layers']):
        if dl not in dst_grads:
            continue
        g = dst_grads[dl]
        valid_mask = ~torch.isnan(g).any(dim=1)
        if valid_mask.sum() == 0:
            continue
        g_valid = g[valid_mask]
        for sl in range(dl):
            for sh in range(cfg['n_heads']):
                d_valid = src_delta_dmodel[(sl, sh)][valid_mask]
                imp_lookup[(sl, sh, dl)] = float((g_valid * d_valid).sum(dim=1).mean().detach())
    log(f'    Importance lookup: {len(imp_lookup)} combos')

    # Step 3: Assemble rows (dict lookups only)
    imp_rows, pert_rows, pv_rows = [], [], []
    nan_pv = {'pert_resample': float('nan'), 'pert_mean': float('nan'), 'pert_zero': float('nan')}
    for _, edge in edges_df.iterrows():
        eid = int(edge['edge_id'])
        sl, sh, dl = int(edge['src_layer']), int(edge['src_head']), int(edge['dst_layer'])
        imp_rows.append({'edge_id': eid, 'imp_value': imp_lookup.get((sl, sh, dl), float('nan'))})
        pert_rows.append({'edge_id': eid, 'pert_value': src_pert.get((sl, sh), float('nan'))})
        pv_rows.append({'edge_id': eid, **src_pv.get((sl, sh), nan_pv)})
    log(f'    Edge rows: {len(imp_rows)} built')

    del dst_grads, src_delta_dmodel, W_O
    gc.collect()

    # Save importance
    imp_df = edges_df.merge(pd.DataFrame(imp_rows), on='edge_id')
    imp_df['importance_metric'] = 'dst_resid_pre_grad_dot_src_delta'
    imp_df['importance_approximation'] = (
        'dst_resid_pre_grad_dot_src_delta')
    imp_df['n_trials'] = n_prompts
    imp_df.to_csv(
        os.path.join(phase_dir, 'importance_per_edge.csv'), index=False)

    imp_abs_df = imp_df.copy()
    imp_abs_df['imp_value'] = imp_abs_df['imp_value'].abs()
    imp_abs_df.to_csv(
        os.path.join(phase_dir, 'importance_per_edge_abs.csv'),
        index=False)

    # Save perturbation
    pert_df = edges_df.merge(pd.DataFrame(pert_rows), on='edge_id')
    pert_df['perturbation_metric'] = 'source_head_L2_dmodel'
    pert_df['n_trials'] = n_prompts
    pert_df.to_csv(
        os.path.join(phase_dir, 'perturbation_per_edge.csv'),
        index=False)

    # Save variants + correlations
    pv_df = pd.DataFrame(pv_rows)
    pv_df.to_csv(
        os.path.join(phase_dir, 'perturbation_variants.csv'),
        index=False)
    rho_rm, p_rm = spearmanr(
        pv_df['pert_resample'], pv_df['pert_mean'])
    rho_rz, p_rz = spearmanr(
        pv_df['pert_resample'], pv_df['pert_zero'])
    rho_mz, p_mz = spearmanr(
        pv_df['pert_mean'], pv_df['pert_zero'])
    corr = {
        'resample_vs_mean': {
            'rho': round(rho_rm, 4),
            'p': float(f'{p_rm:.4e}')},
        'resample_vs_zero': {
            'rho': round(rho_rz, 4),
            'p': float(f'{p_rz:.4e}')},
        'mean_vs_zero': {
            'rho': round(rho_mz, 4),
            'p': float(f'{p_mz:.4e}')},
        'n_edges': n_edges,
    }
    with open(os.path.join(
            phase_dir, 'perturbation_variant_correlations.json'),
            'w') as f:
        json.dump(corr, f, indent=2)

    # Cell classification (signed + abs)
    pert_scores = pd.DataFrame(pert_rows)
    for variant, out_name in [
        ('signed', 'cell_classification.csv'),
        ('abs', 'cell_classification_abs.csv'),
    ]:
        work = imp_df[[
            'edge_id', 'src_layer', 'src_head',
            'dst_layer', 'dst_head', 'imp_value',
            'src_in_gt', 'dst_in_gt',
            'src_gt_subclass', 'dst_gt_subclass',
        ]].copy()
        if variant == 'abs':
            work['imp_value'] = work['imp_value'].abs()
        work = work.merge(pert_scores, on='edge_id')
        work['cell'] = classify_cells(
            work['imp_value'].values, work['pert_value'].values)
        work['importance_metric'] = (
            'dst_resid_pre_grad_dot_src_delta')
        work.to_csv(os.path.join(phase_dir, out_name), index=False)
        counts = dict(pd.Series(work['cell']).value_counts())
        log(f'    {variant}: {counts}')

    # Imp-pert correlation
    valid_imp = imp_df['imp_value'].dropna()
    if len(valid_imp) > 2:
        valid_pert = pert_df.loc[valid_imp.index, 'pert_value']
        rho_ip, _ = spearmanr(valid_imp.abs(), valid_pert)
        log(f'  |imp| ~ pert: rho={rho_ip:.4f}')
    else:
        rho_ip = float('nan')

    config = {
        'phase': '20_scoring', 'model': model_name,
        'importance_method': 'dst_resid_pre_grad_dot_src_delta',
        'perturbation_method': 'source_head_L2_dmodel',
        'n_edges': n_edges, 'n_prompts': n_prompts,
        'grad_batch_size': CACHE_BATCH,
        'imp_pert_rho': (
            round(rho_ip, 4) if not np.isnan(rho_ip) else None),
        'completed_at': datetime.now().isoformat(),
        'elapsed_sec': round(time.time() - t0, 1),
    }
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    log(f'  Phase 20 complete: {time.time() - t0:.1f}s')


log('Cell 5 defined: Phase 20 (Scoring).')

In [ ]:
# Cell 6: Phase 30 -- Dose Response (batched intervention)

def _run_hook_z_intervention_batch(
    model, texts, src_heads_to_patch, clean_z, corrupt_z,
    io_ids_batch, s_ids_batch, prompt_indices, device,
):
    """Batched hook_z replacement. Returns list of task scores."""
    tokens = model.to_tokens(texts, prepend_bos=True).to(device)
    batch_size = tokens.shape[0]

    patches_by_layer = defaultdict(list)
    for (sl, sh) in src_heads_to_patch:
        patches_by_layer[sl].append(sh)

    def make_hook(layer, heads, p_indices):
        def hook_fn(value, hook):
            for b_idx, p_idx in enumerate(p_indices):
                for h in heads:
                    value[b_idx, -1, h, :] = (
                        corrupt_z[layer][p_idx, h, :].to(device))
            return value
        return hook_fn

    fwd_hooks = []
    for layer, heads in patches_by_layer.items():
        hook_name = f'blocks.{layer}.attn.hook_z'
        fwd_hooks.append((
            hook_name,
            make_hook(layer, heads, prompt_indices)))

    with torch.no_grad():
        logits = model.run_with_hooks(tokens, fwd_hooks=fwd_hooks)

    scores = []
    for b in range(batch_size):
        scores.append(float(
            logits[b, -1, io_ids_batch[b]]
            - logits[b, -1, s_ids_batch[b]]))
    return scores


def run_phase_30(model_name, model):
    """Phase 30: Dose response -- 9 groups, ratio sweep.
    Batched intervention for speed.
    k = ceil(total_edges * ratio), NOT pool-relative."""
    cfg = MODEL_REGISTRY[model_name]
    phase_dir = os.path.join(DRIVE_BASE, '30_patching', model_name)
    os.makedirs(phase_dir, exist_ok=True)
    coll_dir = os.path.join(DRIVE_BASE, '10_collection', model_name)
    score_dir = os.path.join(DRIVE_BASE, '20_scoring', model_name)
    config_path = os.path.join(phase_dir, 'config.json')

    log(f'=== Phase 30: Dose Response [{model_name}] ===')
    t0 = time.time()

    if os.path.exists(config_path):
        log(f'  Phase 30 already complete, skipping.')
        return

    prompts_df = pd.read_csv(os.path.join(coll_dir, 'prompts.csv'))
    edges_df = pd.read_csv(
        os.path.join(coll_dir, 'candidate_edges.csv'))
    clean_z = torch.load(
        os.path.join(coll_dir, 'clean_hook_z.pt'),
        map_location=DEVICE)
    corrupt_z = torch.load(
        os.path.join(coll_dir, 'corrupt_hook_z.pt'),
        map_location=DEVICE)
    task_data = torch.load(
        os.path.join(coll_dir, 'task_scores.pt'),
        map_location='cpu')
    io_ids = task_data['io_ids']
    s_ids = task_data['s_ids']
    clean_scores = task_data['scores']

    edge_src = {
        int(r['edge_id']): (int(r['src_layer']), int(r['src_head']))
        for _, r in edges_df.iterrows()}
    total_edges = len(edges_df)
    n_prompts = len(prompts_df)
    all_texts = list(prompts_df['clean_text'])

    for variant, cell_file, out_file in [
        ('signed', 'cell_classification.csv',
         'dose_response_v2.csv'),
        ('abs', 'cell_classification_abs.csv',
         'dose_response_v2_abs.csv'),
    ]:
        cell_df = pd.read_csv(
            os.path.join(score_dir, cell_file))
        orderings = build_orderings(cell_df)
        results = []

        for ratio in RATIOS:
            k = max(1, int(np.ceil(total_edges * ratio)))
            for gname, edge_ids in orderings.items():
                k_actual = min(k, len(edge_ids))
                src_heads = set()
                for eid in edge_ids[:k_actual]:
                    if eid in edge_src:
                        src_heads.add(edge_src[eid])

                # Batched intervention
                for bs in range(
                    0, n_prompts, INTERVENTION_BATCH
                ):
                    be = min(bs + INTERVENTION_BATCH, n_prompts)
                    batch_texts = all_texts[bs:be]
                    batch_io = io_ids[bs:be]
                    batch_s = s_ids[bs:be]
                    p_indices = list(range(bs, be))
                    try:
                        patched = (
                            _run_hook_z_intervention_batch(
                                model, batch_texts, src_heads,
                                clean_z, corrupt_z,
                                batch_io, batch_s,
                                p_indices, DEVICE))
                        for i, p_idx in enumerate(
                            range(bs, be)
                        ):
                            delta = (
                                patched[i] - clean_scores[p_idx])
                            results.append({
                                'group': gname,
                                'ratio': ratio,
                                'k': k,
                                'k_actual': k_actual,
                                'trial_id': p_idx,
                                'delta_target_signed':
                                    round(delta, 6),
                                'delta_target_abs':
                                    round(abs(delta), 6),
                            })
                    except Exception as e:
                        log(f'    WARNING: intervention '
                            f'g={gname} r={ratio} '
                            f'bs={bs}: {e}')
                        for p_idx in range(bs, be):
                            results.append({
                                'group': gname,
                                'ratio': ratio,
                                'k': k,
                                'k_actual': k_actual,
                                'trial_id': p_idx,
                                'delta_target_signed':
                                    float('nan'),
                                'delta_target_abs':
                                    float('nan'),
                            })

            # Incremental save per ratio
            pd.DataFrame(results).to_csv(
                os.path.join(phase_dir, out_file), index=False)
            log(f'    [{variant}] ratio={ratio:.2f} '
                f'({len(results)} rows)')

        # CD ratio summary
        df = pd.DataFrame(results)
        summary = df.groupby(['group', 'ratio']).agg(
            mean_delta=('delta_target_signed', 'mean'),
            mean_abs_delta=('delta_target_abs', 'mean'),
            std_delta=('delta_target_signed', 'std'),
            n=('delta_target_signed', 'count'),
        ).reset_index()
        suffix = '_abs' if variant == 'abs' else ''
        summary.to_csv(
            os.path.join(phase_dir,
                         f'cd_ratio_summary{suffix}.csv'),
            index=False)

    del clean_z, corrupt_z
    gc.collect()
    torch.cuda.empty_cache()

    config = {
        'phase': '30_patching', 'model': model_name,
        'intervention_type':
            'source_head_hook_z_replacement',
        'exact_edge_isolation': False,
        'ratios': RATIOS,
        'n_prompts': len(prompts_df),
        'total_edges': total_edges,
        'intervention_batch_size': INTERVENTION_BATCH,
        'completed_at': datetime.now().isoformat(),
        'elapsed_sec': round(time.time() - t0, 1),
    }
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    log(f'  Phase 30 complete: {time.time() - t0:.1f}s')


log('Cell 6 defined: Phase 30 (Dose Response).')

In [ ]:
# Cell 7: Phase 35 -- Recoverability / Utility

def run_phase_35(model_name, model):
    """Phase 35: Recoverability -- threshold rules, baselines,
    faithfulness.

    GT-free for all Stage 1 models: report threshold separation
    + selected_edges only."""
    cfg = MODEL_REGISTRY[model_name]
    has_gt = cfg['has_gt_heads']
    n_layers = cfg['n_layers']
    n_heads = cfg['n_heads']

    phase_dir = os.path.join(
        DRIVE_BASE, '35_recoverability', model_name)
    os.makedirs(phase_dir, exist_ok=True)
    score_dir = os.path.join(DRIVE_BASE, '20_scoring', model_name)
    coll_dir = os.path.join(
        DRIVE_BASE, '10_collection', model_name)
    config_path = os.path.join(phase_dir, 'config.json')

    log(f'=== Phase 35: Recoverability [{model_name}] ===')
    t0 = time.time()

    if os.path.exists(config_path):
        log(f'  Phase 35 already complete, skipping.')
        return

    gt_set = set()

    # --- Threshold Recovery ---
    threshold_rows = []
    for baseline, cell_file in BASELINES:
        cell_df = pd.read_csv(
            os.path.join(score_dir, cell_file))
        if 'celld' in baseline:
            null_pool = cell_df[
                cell_df['cell'] == 'D']['imp_value'].abs()
        else:
            null_pool = cell_df[
                cell_df['cell'].isin(
                    ['C', 'D'])]['imp_value'].abs()
        imp_all = cell_df['imp_value'].abs()

        for rule_name, rule_fn in THRESHOLD_RULES.items():
            threshold = rule_fn(null_pool)
            selected = cell_df[imp_all >= threshold]
            n_sel = len(selected)
            src_h = set(zip(
                selected['src_layer'].astype(int),
                selected['src_head'].astype(int)))
            dst_h = set(zip(
                selected['dst_layer'].astype(int),
                selected['dst_head'].astype(int)))
            src_gt = (len(src_h & gt_set) / len(gt_set)
                      if gt_set else float('nan'))
            dst_gt = (len(dst_h & gt_set) / len(gt_set)
                      if gt_set else float('nan'))
            all_t = src_h | dst_h
            if gt_set and all_t:
                prec = len(all_t & gt_set) / len(all_t)
                rec = len(all_t & gt_set) / len(gt_set)
                f1 = (2 * prec * rec / (prec + rec)
                      if (prec + rec) > 0 else 0)
            else:
                prec = rec = f1 = float('nan')

            threshold_rows.append({
                'baseline': baseline, 'rule': rule_name,
                'threshold': round(threshold, 6),
                'selected_edges': n_sel,
                'selected_fraction': round(
                    n_sel / len(cell_df), 4)
                    if len(cell_df) > 0 else 0,
                'src_gt_coverage': round(src_gt, 4)
                    if not np.isnan(src_gt) else float('nan'),
                'dst_gt_coverage': round(dst_gt, 4)
                    if not np.isnan(dst_gt) else float('nan'),
                'recall_proxy': round(rec, 4)
                    if not np.isnan(rec) else float('nan'),
                'precision_proxy': round(prec, 4)
                    if not np.isnan(prec) else float('nan'),
                'f1_proxy': round(f1, 4)
                    if not np.isnan(f1) else float('nan'),
            })

    threshold_df = pd.DataFrame(threshold_rows)
    threshold_df.to_csv(
        os.path.join(phase_dir, 'threshold_recovery.csv'),
        index=False)
    log(f'  Threshold recovery: {len(threshold_df)} rows')
    log(threshold_df[[
        'baseline', 'rule', 'threshold',
        'selected_edges', 'selected_fraction',
    ]].to_string(index=False))

    # --- Faithfulness vs Budget (batched) ---
    faithfulness_rows = []
    clean_z = torch.load(
        os.path.join(coll_dir, 'clean_hook_z.pt'),
        map_location=DEVICE)
    corrupt_z = torch.load(
        os.path.join(coll_dir, 'corrupt_hook_z.pt'),
        map_location=DEVICE)
    prompts_df = pd.read_csv(
        os.path.join(coll_dir, 'prompts.csv'))
    task_data = torch.load(
        os.path.join(coll_dir, 'task_scores.pt'),
        map_location='cpu')
    io_ids = task_data['io_ids']
    s_ids = task_data['s_ids']
    clean_scores = task_data['scores']
    edges_df = pd.read_csv(
        os.path.join(coll_dir, 'candidate_edges.csv'))
    edge_src = {
        int(r['edge_id']):
            (int(r['src_layer']), int(r['src_head']))
        for _, r in edges_df.iterrows()}
    total_edges = len(edges_df)
    n_prompts = len(prompts_df)
    all_texts = list(prompts_df['clean_text'])
    all_heads = set(
        (l, h)
        for l in range(n_layers)
        for h in range(n_heads))

    for baseline, cell_file in BASELINES:
        cell_df = pd.read_csv(
            os.path.join(score_dir, cell_file))
        if 'celld' in baseline:
            null_pool = cell_df[
                cell_df['cell'] == 'D']['imp_value'].abs()
        else:
            null_pool = cell_df[
                cell_df['cell'].isin(
                    ['C', 'D'])]['imp_value'].abs()
        cell_sorted = cell_df.copy()
        cell_sorted['abs_imp'] = (
            cell_sorted['imp_value'].abs())
        cell_sorted = cell_sorted.sort_values(
            'abs_imp', ascending=False)

        for rule_name, rule_fn in THRESHOLD_RULES.items():
            threshold = rule_fn(null_pool)
            above = cell_sorted[
                cell_sorted['abs_imp'] >= threshold]

            for budget in FAITHFULNESS_BUDGETS:
                selected = above.head(budget)
                n_sel = len(selected)
                if n_sel == 0:
                    faithfulness_rows.append({
                        'baseline': baseline,
                        'rule': rule_name,
                        'budget': budget,
                        'selected_edges': 0,
                        'faithfulness': float('nan'),
                        'sparsity': 1.0,
                        'src_gt_coverage': float('nan'),
                        'dst_gt_coverage': float('nan'),
                    })
                    continue

                sel_src = set()
                for eid in selected['edge_id']:
                    if int(eid) in edge_src:
                        sel_src.add(edge_src[int(eid)])
                heads_to_corrupt = all_heads - sel_src

                # Batched faithfulness
                faith_scores = []
                for bs in range(
                    0, n_prompts, INTERVENTION_BATCH
                ):
                    be = min(
                        bs + INTERVENTION_BATCH, n_prompts)
                    try:
                        batch_s = (
                            _run_hook_z_intervention_batch(
                                model,
                                all_texts[bs:be],
                                heads_to_corrupt,
                                clean_z, corrupt_z,
                                io_ids[bs:be],
                                s_ids[bs:be],
                                list(range(bs, be)),
                                DEVICE))
                        faith_scores.extend(batch_s)
                    except Exception:
                        faith_scores.extend(
                            [float('nan')] * (be - bs))

                valid = [
                    (f, c)
                    for f, c in zip(
                        faith_scores, clean_scores)
                    if not np.isnan(f)]
                if valid:
                    mf = np.mean([f for f, _ in valid])
                    mc = np.mean([c for _, c in valid])
                    faith = (mf / mc
                             if abs(mc) > 1e-8
                             else float('nan'))
                else:
                    faith = float('nan')

                sparsity = 1.0 - n_sel / total_edges
                sh = set(zip(
                    selected['src_layer'].astype(int),
                    selected['src_head'].astype(int)))
                dh = set(zip(
                    selected['dst_layer'].astype(int),
                    selected['dst_head'].astype(int)))
                sg = (len(sh & gt_set) / len(gt_set)
                      if gt_set else float('nan'))
                dg = (len(dh & gt_set) / len(gt_set)
                      if gt_set else float('nan'))

                faithfulness_rows.append({
                    'baseline': baseline,
                    'rule': rule_name,
                    'budget': budget,
                    'selected_edges': n_sel,
                    'faithfulness': round(faith, 6)
                        if not np.isnan(faith)
                        else float('nan'),
                    'sparsity': round(sparsity, 6),
                    'src_gt_coverage': round(sg, 4)
                        if not np.isnan(sg)
                        else float('nan'),
                    'dst_gt_coverage': round(dg, 4)
                        if not np.isnan(dg)
                        else float('nan'),
                })

            # Incremental save
            pd.DataFrame(faithfulness_rows).to_csv(
                os.path.join(
                    phase_dir, 'faithfulness_vs_budget.csv'),
                index=False)

        log(f'    baseline={baseline} faithfulness done')

    del clean_z, corrupt_z
    gc.collect()
    torch.cuda.empty_cache()

    # Budget-matched comparison
    faith_df = pd.DataFrame(faithfulness_rows)
    bmc_rows = []
    for budget in FAITHFULNESS_BUDGETS:
        for bl, _ in BASELINES:
            sub = faith_df[
                (faith_df['budget'] == budget) &
                (faith_df['baseline'] == bl)]
            if len(sub) > 0:
                best = sub.sort_values(
                    'faithfulness', ascending=False).iloc[0]
                bmc_rows.append({
                    'budget': budget,
                    'baseline': bl,
                    'faithfulness': best['faithfulness'],
                    'sparsity': best['sparsity'],
                    'selected_edges': best['selected_edges'],
                    'src_gt_coverage':
                        best['src_gt_coverage'],
                    'dst_gt_coverage':
                        best['dst_gt_coverage'],
                })
    pd.DataFrame(bmc_rows).to_csv(
        os.path.join(
            phase_dir, 'budget_matched_comparison.csv'),
        index=False)

    # False positive analysis
    fp_rows = []
    for baseline, cell_file in BASELINES:
        cell_df = pd.read_csv(
            os.path.join(score_dir, cell_file))
        if 'celld' in baseline:
            np_ = cell_df[
                cell_df['cell'] == 'D']['imp_value'].abs()
        else:
            np_ = cell_df[
                cell_df['cell'].isin(
                    ['C', 'D'])]['imp_value'].abs()
        imp_a = cell_df['imp_value'].abs()
        for rn, rf in THRESHOLD_RULES.items():
            thr = rf(np_)
            sel = cell_df[imp_a >= thr]
            for _, row in sel.iterrows():
                si = int((int(row['src_layer']),
                          int(row['src_head'])) in gt_set)
                di = int((int(row['dst_layer']),
                          int(row['dst_head'])) in gt_set)
                if not si and not di:
                    ft = 'neither_gt'
                elif si and not di:
                    ft = 'src_only_gt'
                elif not si and di:
                    ft = 'dst_only_gt'
                else:
                    ft = 'both_gt'
                fp_rows.append({
                    'baseline': baseline, 'rule': rn,
                    'edge_id': int(row['edge_id']),
                    'src_in_gt': si, 'dst_in_gt': di,
                    'src_cell': row.get('cell', ''),
                    'dst_cell': '', 'fp_type': ft,
                })
    if fp_rows:
        pd.DataFrame(fp_rows).to_csv(
            os.path.join(
                phase_dir, 'false_positive_analysis.csv'),
            index=False)

    # Recovery summary
    summary = {
        'model': model_name, 'has_gt_heads': has_gt,
        'baselines': [b for b, _ in BASELINES],
        'rules': list(THRESHOLD_RULES.keys()),
        'threshold_contrasts': {},
        'approximation_note':
            'source_head_hook_z_replacement, '
            'not exact edge isolation',
    }
    for rn in THRESHOLD_RULES:
        sr = threshold_df[
            (threshold_df['baseline'] == 'standard') &
            (threshold_df['rule'] == rn)]
        cr = threshold_df[
            (threshold_df['baseline'] == 'celld') &
            (threshold_df['rule'] == rn)]
        if len(sr) > 0 and len(cr) > 0:
            summary['threshold_contrasts'][rn] = {
                'standard_threshold':
                    float(sr['threshold'].iloc[0]),
                'celld_threshold':
                    float(cr['threshold'].iloc[0]),
                'standard_selected':
                    int(sr['selected_edges'].iloc[0]),
                'celld_selected':
                    int(cr['selected_edges'].iloc[0]),
            }

    # absLD gap closure
    sr2 = threshold_df[
        threshold_df['baseline'] == 'standard']
    ar2 = threshold_df[
        threshold_df['baseline'] == 'absld']
    if len(sr2) > 0 and len(ar2) > 0:
        summary['absld_gap_closure'] = {
            'standard_mean_threshold': round(
                float(sr2['threshold'].mean()), 6),
            'absld_mean_threshold': round(
                float(ar2['threshold'].mean()), 6),
        }

    with open(
        os.path.join(phase_dir, 'recovery_summary.json'),
        'w',
    ) as f:
        json.dump(summary, f, indent=2)

    config = {
        'phase': '35_recoverability',
        'model': model_name,
        'has_gt_heads': has_gt,
        'budgets': FAITHFULNESS_BUDGETS,
        'intervention_batch_size': INTERVENTION_BATCH,
        'completed_at': datetime.now().isoformat(),
        'elapsed_sec': round(time.time() - t0, 1),
    }
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    log(f'  Phase 35 complete: {time.time() - t0:.1f}s')


log('Cell 7 defined: Phase 35 (Recoverability).')

In [ ]:
# Cell 8: Phase 36 -- Discovery Summaries

def run_phase_36(model_name):
    """Phase 36: Discovery summaries -- selected edge inventories.
    No model needed -- pure CSV analysis."""
    cfg = MODEL_REGISTRY[model_name]
    has_gt = cfg['has_gt_heads']

    phase_dir = os.path.join(
        DRIVE_BASE, '36_discovery', model_name)
    os.makedirs(phase_dir, exist_ok=True)
    score_dir = os.path.join(
        DRIVE_BASE, '20_scoring', model_name)
    config_path = os.path.join(phase_dir, 'config.json')

    log(f'=== Phase 36: Discovery [{model_name}] ===')
    t0 = time.time()

    if os.path.exists(config_path):
        log(f'  Phase 36 already complete, skipping.')
        return

    gt_set = set()

    for baseline, cell_file in BASELINES:
        cell_df = pd.read_csv(
            os.path.join(score_dir, cell_file))
        if 'celld' in baseline:
            null_pool = cell_df[
                cell_df['cell'] == 'D']['imp_value'].abs()
        else:
            null_pool = cell_df[
                cell_df['cell'].isin(
                    ['C', 'D'])]['imp_value'].abs()
        imp_all = cell_df['imp_value'].abs()

        best_rule = None
        best_score = float('-inf')
        best_threshold = float('nan')
        best_selected = 0
        best_src = float('nan')
        best_dst = float('nan')

        for rn, rf in THRESHOLD_RULES.items():
            thr = rf(null_pool)
            sel = cell_df[imp_all >= thr].copy()
            sel.to_csv(
                os.path.join(
                    phase_dir,
                    f'selected_edges_{baseline}_{rn}.csv'),
                index=False)

            src_h = set(zip(
                sel['src_layer'].astype(int),
                sel['src_head'].astype(int)))
            dst_h = set(zip(
                sel['dst_layer'].astype(int),
                sel['dst_head'].astype(int)))

            if gt_set:
                sc = len(src_h & gt_set) / len(gt_set)
                dc = len(dst_h & gt_set) / len(gt_set)
                at = src_h | dst_h
                pr = (len(at & gt_set) / len(at)
                      if at else 0)
                rc = len(at & gt_set) / len(gt_set)
                f1 = (2 * pr * rc / (pr + rc)
                      if (pr + rc) > 0 else 0)
                proxy = f1
            else:
                sc = dc = float('nan')
                # GT-free: prefer lower threshold
                proxy = -thr

            if proxy > best_score:
                best_score = proxy
                best_rule = rn
                best_threshold = thr
                best_selected = len(sel)
                best_src = sc
                best_dst = dc

        disc = {
            'baseline': baseline,
            'best_rule': best_rule,
            'threshold': round(best_threshold, 6),
            'selected_edges': best_selected,
            'src_gt_coverage': round(best_src, 4)
                if not np.isnan(best_src) else None,
            'dst_gt_coverage': round(best_dst, 4)
                if not np.isnan(best_dst) else None,
            'has_gt': has_gt,
            'notes': ('GT-free: threshold separation only'
                      if not has_gt else 'GT available'),
        }
        with open(
            os.path.join(
                phase_dir,
                f'discovery_summary_{baseline}.json'),
            'w',
        ) as f:
            json.dump(disc, f, indent=2)
        log(f'    {baseline}: best={best_rule}, '
            f'sel={best_selected}, '
            f'thr={best_threshold:.6f}')

        # Canonical copy (mean_2.0sd)
        import shutil
        can_src = os.path.join(
            phase_dir,
            f'selected_edges_{baseline}_mean_2.0sd.csv')
        can_dst = os.path.join(
            phase_dir,
            f'selected_edges_{baseline}.csv')
        if os.path.exists(can_src):
            shutil.copy2(can_src, can_dst)

    config = {
        'phase': '36_discovery', 'model': model_name,
        'completed_at': datetime.now().isoformat(),
        'elapsed_sec': round(time.time() - t0, 1),
    }
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    log(f'  Phase 36 complete: {time.time() - t0:.1f}s')


log('Cell 8 defined: Phase 36 (Discovery Summaries).')

In [ ]:
# Cell 9: Aggregate Summaries

def run_aggregate():
    """Aggregate results across all completed models."""
    agg_dir = os.path.join(DRIVE_BASE, '50_aggregate')
    os.makedirs(agg_dir, exist_ok=True)
    log('=== Aggregate Summaries ===')

    policy_rows = []
    model_rows = []

    for model_name in MODEL_ORDER:
        dd = os.path.join(
            DRIVE_BASE, '36_discovery', model_name)
        rd = os.path.join(
            DRIVE_BASE, '35_recoverability', model_name)
        if not os.path.exists(
            os.path.join(dd, 'config.json')
        ):
            log(f'  {model_name}: not complete, skip.')
            continue

        tp = os.path.join(rd, 'threshold_recovery.csv')
        thresh_df = (pd.read_csv(tp)
                     if os.path.exists(tp)
                     else pd.DataFrame())

        for bl in [
            'standard', 'celld', 'absld', 'absld_celld'
        ]:
            dp = os.path.join(
                dd, f'discovery_summary_{bl}.json')
            if os.path.exists(dp):
                with open(dp) as f:
                    disc = json.load(f)
                policy_rows.append({
                    'model_name': model_name,
                    'condition': bl,
                    'best_threshold_rule':
                        disc.get('best_rule', ''),
                    'threshold':
                        disc.get('threshold', float('nan')),
                    'selected_edges':
                        disc.get('selected_edges', 0),
                    'faithfulness': float('nan'),
                    'sparsity': float('nan'),
                    'src_gt_coverage':
                        disc.get(
                            'src_gt_coverage', float('nan')),
                    'dst_gt_coverage':
                        disc.get(
                            'dst_gt_coverage', float('nan')),
                })

        # Fill faithfulness from budget_matched
        bmc_p = os.path.join(
            rd, 'budget_matched_comparison.csv')
        if os.path.exists(bmc_p):
            bmc = pd.read_csv(bmc_p)
            for row in policy_rows:
                if row['model_name'] == model_name:
                    m = bmc[
                        bmc['baseline'] == row['condition']]
                    if len(m) > 0:
                        b = m.sort_values(
                            'faithfulness',
                            ascending=False).iloc[0]
                        row['faithfulness'] = b.get(
                            'faithfulness', float('nan'))
                        row['sparsity'] = b.get(
                            'sparsity', float('nan'))

        # Gate 1 check
        gate1 = False
        dose_p = os.path.join(
            DRIVE_BASE, '30_patching', model_name,
            'cd_ratio_summary.csv')
        if os.path.exists(dose_p):
            dose = pd.read_csv(dose_p)
            ci = dose[
                dose['group'] == 'C_imp_asc'
            ]['mean_abs_delta'].mean()
            dp_ = dose[
                dose['group'] == 'D_pert_asc'
            ]['mean_abs_delta'].mean()
            gate1 = bool(ci > dp_) if not np.isnan(ci) else False

        # Threshold gaps
        def _mean_thresh(bl_name):
            if len(thresh_df) == 0:
                return float('nan')
            s = thresh_df[
                thresh_df['baseline'] == bl_name]
            return (float(s['threshold'].mean())
                    if len(s) > 0 else float('nan'))

        st = _mean_thresh('standard')
        ct = _mean_thresh('celld')
        at = _mean_thresh('absld')
        act = _mean_thresh('absld_celld')

        def _safe_diff(a, b):
            if np.isnan(a) or np.isnan(b):
                return float('nan')
            return round(a - b, 6)

        model_rows.append({
            'model_name': model_name,
            'gate1_passed': gate1,
            'standard_vs_celld_gap': _safe_diff(st, ct),
            'standard_vs_absld_gap': _safe_diff(st, at),
            'absld_vs_absld_celld_gap': _safe_diff(at, act),
            'main_mechanism': 'pending',
            'recommended_interpretation': 'pending',
        })

    if policy_rows:
        pd.DataFrame(policy_rows).to_csv(
            os.path.join(
                agg_dir, 'aggregate_policy_summary.csv'),
            index=False)
    if model_rows:
        pd.DataFrame(model_rows).to_csv(
            os.path.join(
                agg_dir, 'aggregate_model_summary.csv'),
            index=False)
    if policy_rows:
        pd.DataFrame(policy_rows).to_csv(
            os.path.join(
                agg_dir, 'appendix_ready_tables.csv'),
            index=False)

    log(f'  Aggregate: {len(policy_rows)} policy, '
        f'{len(model_rows)} model rows.')


log('Cell 9 defined: Aggregate summaries.')

In [ ]:
# Cell 10: Main Execution Loop -- All 4 Shipped Models, All Phases

# Generate shared prompts
prompts_df = generate_ioi_prompts(N_PROMPTS, seed=42)
log(f'Generated {len(prompts_df)} IOI prompts.')
log(f'Sample: {prompts_df.iloc[0]["clean_text"]}')

# Save master prompts + registry + config
master_dir = os.path.join(DRIVE_BASE, '10_collection')
os.makedirs(master_dir, exist_ok=True)
prompts_df.to_csv(
    os.path.join(master_dir, 'prompts_master.csv'),
    index=False)

reg_out = {}
for mn, mc in MODEL_REGISTRY.items():
    reg_out[mn] = {
        k: (str(v) if isinstance(v, torch.dtype) else v)
        for k, v in mc.items()}
with open(
    os.path.join(master_dir, 'model_registry.json'), 'w'
) as f:
    json.dump(reg_out, f, indent=2)

with open(
    os.path.join(master_dir, 'config.json'), 'w'
) as f:
    json.dump({
        'task': 'ioi', 'n_prompts': N_PROMPTS,
        'prompt_generation': 'manual with seed=42',
        'candidate_edge_policy':
            'src_layer < dst_layer, all heads',
        'edge_restriction_policy':
            'none (no source-head filtering)',
        'ratio_grid': RATIOS,
        'threshold_rules': list(THRESHOLD_RULES.keys()),
        'budget_grid': FAITHFULNESS_BUDGETS,
    }, f, indent=2)

nb_dir = os.path.join(DRIVE_BASE, '00_notebook')
os.makedirs(nb_dir, exist_ok=True)
with open(
    os.path.join(nb_dir, 'notebook_metadata.json'), 'w'
) as f:
    json.dump({
        'notebook': 'acdc_ioi_cross_model_stage1',
        'stage': 1,
        'models': MODEL_ORDER,
        'n_prompts': N_PROMPTS,
        'created_at': datetime.now().isoformat(),
    }, f, indent=2)

progress = load_progress()
log(f'Progress: {json.dumps(progress, indent=2)}')

# ============================================================
# Main model loop
# ============================================================
overall_t0 = time.time()
skipped_models = []

for model_name in MODEL_ORDER:
    log(f'\n{"="*60}')
    log(f'MODEL: {model_name}')
    log(f'{"="*60}')

    # Skip if fully complete
    cm = progress.get('completed_models', {})
    if model_name in cm:
        if cm[model_name].get('all_phases_done'):
            log(f'  Already complete, skipping.')
            continue

    progress['current_model'] = model_name
    save_progress(progress)

    model = None
    try:
        model = load_tl_model(model_name)
    except Exception as e:
        log(f'  FATAL: load failed: {e}')
        skipped_models.append(
            (model_name, f'load_failed: {e}'))
        continue

    try:
        # Phase 10
        progress['current_phase'] = '10_collection'
        save_progress(progress)
        solvable = run_phase_10(
            model_name, model, prompts_df)

        if not solvable:
            log(f'  IOI not solvable. Skipping 20-36.')
            skipped_models.append(
                (model_name, 'ioi_not_solvable'))
            progress.setdefault(
                'completed_models', {})[model_name] = {
                'solvable': False,
                'all_phases_done': False}
            save_progress(progress)
            unload_model(model)
            model = None
            continue

        # Phase 20
        progress['current_phase'] = '20_scoring'
        save_progress(progress)
        run_phase_20(model_name, model)

        # Phase 30
        progress['current_phase'] = '30_patching'
        save_progress(progress)
        run_phase_30(model_name, model)

        # Phase 35
        progress['current_phase'] = '35_recoverability'
        save_progress(progress)
        run_phase_35(model_name, model)

        # Unload before Phase 36 (no model needed)
        unload_model(model)
        model = None

        # Phase 36
        progress['current_phase'] = '36_discovery'
        save_progress(progress)
        run_phase_36(model_name)

        progress.setdefault(
            'completed_models', {})[model_name] = {
            'solvable': True,
            'all_phases_done': True,
            'completed_at': datetime.now().isoformat()}
        save_progress(progress)
        log(f'  All phases complete for {model_name}.')

    except Exception as e:
        log(f'  ERROR: {e}')
        import traceback
        traceback.print_exc()
        skipped_models.append(
            (model_name, f'runtime_error: {e}'))
        progress.setdefault(
            'completed_models', {})[model_name] = {
            'error': str(e),
            'all_phases_done': False,
            'failed_phase': progress.get(
                'current_phase', 'unknown')}
        save_progress(progress)

    finally:
        if model is not None:
            try:
                unload_model(model)
            except Exception:
                pass
            model = None
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        print_vram('[between models]')

# ============================================================
# Aggregate
# ============================================================
run_aggregate()

manifest = {
    'notebook': 'acdc_ioi_cross_model_stage1',
    'task': 'ioi', 'stage': 1,
    'models': MODEL_ORDER,
    'n_prompts': N_PROMPTS, 'seed': 42,
    'drive_root': DRIVE_BASE,
    'started_at': datetime.fromtimestamp(
        overall_t0).isoformat(),
    'ended_at': datetime.now().isoformat(),
    'total_elapsed_sec': round(
        time.time() - overall_t0, 1),
    'skipped_models': skipped_models,
    'completed_models': progress.get(
        'completed_models', {}),
}
with open(
    os.path.join(nb_dir, 'run_manifest.json'), 'w'
) as f:
    json.dump(manifest, f, indent=2)
with open(
    os.path.join(DRIVE_BASE, 'record.json'), 'w'
) as f:
    json.dump(manifest, f, indent=2)

log(f'\n{"="*60}')
log('ALL MODELS COMPLETE')
log(f'Total: {time.time() - overall_t0:.0f}s')
if skipped_models:
    log(f'Skipped: {skipped_models}')
log(f'{"="*60}')

In [ ]:
# Cell 11: Verification + Summary

log('=== VERIFICATION ===')

phases = [
    '10_collection', '20_scoring', '30_patching',
    '35_recoverability', '36_discovery',
]

for mn in MODEL_ORDER:
    log(f'\nModel: {mn}')
    for ph in phases:
        cp = os.path.join(DRIVE_BASE, ph, mn, 'config.json')
        if os.path.exists(cp):
            with open(cp) as f:
                c = json.load(f)
            log(f'  {ph}: DONE ({c.get("elapsed_sec", "?")}s)')
        else:
            log(f'  {ph}: MISSING')

# Aggregate summary
agg_dir = os.path.join(DRIVE_BASE, '50_aggregate')
msp = os.path.join(agg_dir, 'aggregate_model_summary.csv')
if os.path.exists(msp):
    log('\n--- Model Summary ---')
    log(pd.read_csv(msp).to_string(index=False))

psp = os.path.join(agg_dir, 'aggregate_policy_summary.csv')
if os.path.exists(psp):
    log('\n--- Policy Summary ---')
    log(pd.read_csv(psp).to_string(index=False))

# Threshold contrasts
for mn in MODEL_ORDER:
    rp = os.path.join(
        DRIVE_BASE, '35_recoverability', mn,
        'recovery_summary.json')
    if os.path.exists(rp):
        with open(rp) as f:
            r = json.load(f)
        tc = r.get('threshold_contrasts', {})
        if tc:
            log(f'\n--- {mn} Thresholds ---')
            for rule, c in tc.items():
                log(f'  {rule}: std={c["standard_threshold"]:.6f}, '
                    f'celld={c["celld_threshold"]:.6f}, '
                    f'std_sel={c["standard_selected"]}, '
                    f'celld_sel={c["celld_selected"]}')

log('\n=== VERIFICATION COMPLETE ===')

In [ ]:
from google.colab import runtime
runtime.unassign()
